# Day 2 + Day 3 on Kaggle: Llama 8B target, Llama-3.2-1B draft (2x T4)

Same experiments as `day2_speculative_hindi` and `day3_trace_simulator`, with the target changed from 3B to 8B. **MorphTok is skipped.**

**Kaggle setup (right sidebar):**
1. *Accelerator* = **GPU T4 x2**
2. *Internet* = **On**
3. *Add-ons > Secrets*: add a secret called `HF_TOKEN` (your Hugging Face token). It needs access to `Meta-Llama-3-8B-Instruct`, `Llama-3.2-1B-Instruct` and the FLORES+ dataset.

**How the 8B fits:** fp16 weights of an 8B model are ~15 GiB, more than one T4 (14.5 GiB). The target is layer-split across both GPUs with `device_map`. The 1B draft lives on GPU 0.

**Notebook layout**
- **Part A (needs the GPU):** 8B greedy translations -> 1B draft traces -> timing / cost ratio `c`. Writes `draft_traces.json`, `timing.json`, `target_outputs.jsonl`.
- **Part B (CPU only):** trace simulator, fixed-k, entropy policy, oracle. Reads the files from Part A.

Set `N = 20` in the config cell first as a smoke test.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before torch is imported
!pip -q install -U transformers accelerate datasets huggingface_hub pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 26.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you hav

In [2]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"Kaggle secret HF_TOKEN not available ({e}); paste your token below")
    login()

Kaggle secret HF_TOKEN not available (Connection error trying to communicate with service.); paste your token below


# Part A: Day 2 (target outputs, draft traces, cost ratio)

In [3]:
import gc, json, time, re
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

TARGET = "meta-llama/Meta-Llama-3-8B-Instruct"   # alt: "meta-llama/Llama-3.1-8B-Instruct" (officially supports Hindi; same family as the 3.2-1B draft)
DRAFT  = "meta-llama/Llama-3.2-1B-Instruct"
DTYPE  = torch.float16          # T4 has no real bf16
N      = None                   # None = all 1012 sentences; use 20 for a smoke test
BATCH, MAX_NEW = 16, 256        # lower BATCH to 8 if generation runs out of memory
dev = "cuda:0"                  # inputs enter the (sharded) target here; the draft also lives here
MAX_MEMORY = {0: "8GiB", 1: "10GiB"}   # cap for the 8B target per GPU (GPU 0 keeps room for the 1B draft + KV cache)
OUT = Path("/kaggle/working/day2_8b"); OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")

def sync_all():
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def gpu_report(tag=""):
    print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i)/2**30:.1f} GiB allocated" for i in range(torch.cuda.device_count())))

tok = AutoTokenizer.from_pretrained(TARGET)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

AssertionError: Need 2 GPUs: Settings > Accelerator > GPU T4 x2

In [ ]:
from datasets import load_dataset
# If gated access isn't approved yet, load the GitHub-release text files instead.
en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split="devtest")
hi_by_id = {r["id"]: r["text"] for r in hi}
rows = [(r["id"], r["text"], hi_by_id[r["id"]]) for r in en]
if N: rows = rows[:N]
ids, src, ref = zip(*rows)
print(len(rows), "sentences\n", src[0], "\n", ref[0])

## 1. Target (8B) greedy translations
The 8B model is loaded sharded across both T4s. Outputs are appended to `target_outputs.jsonl` after every batch, so if the Kaggle session dies you can re-run this cell and it resumes.

In [ ]:
def build_prompt(s):
    msgs = [{"role": "user", "content":
             f"Translate the following English sentence into Hindi. Output only the Hindi translation.\n\n{s}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

prompts = [build_prompt(s) for s in src]

target = AutoModelForCausalLM.from_pretrained(
    TARGET, torch_dtype=DTYPE, device_map="auto", max_memory=MAX_MEMORY, low_cpu_mem_usage=True,
).eval()
print("modules per device:", dict(Counter(target.hf_device_map.values())))
gpu_report("after loading target:")

# stop ids: whatever the generation config says, plus <|eot_id|> (Llama-3 instruct end-of-turn)
eos = target.generation_config.eos_token_id
eos_ids = set(eos if isinstance(eos, list) else [eos])
eos_ids.add(tok.convert_tokens_to_ids("<|eot_id|>"))

OUT_JSONL = OUT / "target_outputs.jsonl"
records = []
if OUT_JSONL.exists():
    records = [json.loads(l) for l in open(OUT_JSONL)]
    if [r["id"] for r in records] != list(ids[:len(records)]):   # stale file from another run / another N
        records = []
    print(f"resuming: {len(records)} / {len(rows)} sentences already done")

with open(OUT_JSONL, "a" if records else "w") as f:
    for i in tqdm(range(len(records), len(prompts), BATCH)):
        enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
        with torch.no_grad():
            out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,
                                  max_new_tokens=MAX_NEW, eos_token_id=list(eos_ids),
                                  pad_token_id=tok.pad_token_id)
        gen = out[:, enc.input_ids.shape[1]:].tolist()
        for j, g in enumerate(gen):
            cut = next((p+1 for p, t in enumerate(g) if t in eos_ids), len(g))   # keep the eos token
            g = g[:cut]
            pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()        # prompt without padding
            rec = dict(id=ids[i+j], src=src[i+j], ref=ref[i+j], prompt_ids=pid,
                       out_ids=g, out_text=tok.decode(g, skip_special_tokens=True).strip())
            records.append(rec); f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

print(f"{len(records)} target outputs saved to {OUT_JSONL}")
for r in records[:3]: print(r["src"], "\n ->", r["out_text"], "\n")
n_hit = sum(len(r["out_ids"]) >= MAX_NEW for r in records)
print(f"{n_hit} outputs hit MAX_NEW={MAX_NEW} without finishing")

## 2. Draft (1B) pass: per-position argmax and entropy (teacher-forced on the 8B output)

In [ ]:
gc.collect(); torch.cuda.empty_cache()
draft = AutoModelForCausalLM.from_pretrained(DRAFT, torch_dtype=DTYPE).to(dev).eval()
gpu_report("after loading draft:")

# draft and target must share a vocabulary, otherwise the whole experiment is meaningless
tok_d = AutoTokenizer.from_pretrained(DRAFT)
assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"
probe = (records[0]["out_text"] + " नमस्ते दुनिया. Hello world!")
assert tok_d(probe, add_special_tokens=False).input_ids == tok(probe, add_special_tokens=False).input_ids, "tokenizers disagree"
print("draft/target tokenizers match")

traces = []
for r in tqdm(records):
    p, o = r["prompt_ids"], r["out_ids"]
    x = torch.tensor([p + o], device=dev)
    with torch.no_grad():
        lg = draft(x).logits[0, len(p)-1:-1].float()   # position t predicts output token t
    lp  = torch.log_softmax(lg, -1)
    ent = -(lp.exp() * lp).sum(-1)                      # entropy in nats
    am  = lp.argmax(-1)
    tg  = torch.tensor(o, device=dev)
    traces.append(dict(id=r["id"], target_ids=o, draft_argmax=am.tolist(), entropy=ent.tolist(),
                       p_target=lp.gather(1, tg[:, None]).squeeze(1).exp().tolist(),
                       match=(am == tg).tolist()))

json.dump(traces, open(OUT/"draft_traces.json", "w"))
m = np.concatenate([t["match"] for t in traces]); e = np.concatenate([t["entropy"] for t in traces])
print(f"positions: {len(m)} | token-level draft/target agreement: {m.mean():.3f} | mean entropy: {e.mean():.3f} nats")
print(f"mean entropy when match: {e[m].mean():.3f} | when mismatch: {e[~m].mean():.3f}")

## 3. Forward-pass timing and cost ratio c
Batch size 1, KV cache, median over many steps. `q` is how many tokens go in per call (1 = decode step, gamma+1 = verification).

Setup caveat for the write-up: the 8B target is layer-split over two T4s (sequential, not parallel), so every decode step includes a GPU0 -> GPU1 hop, while the draft runs on one GPU. State this when you quote `c`.

In [ ]:
@torch.no_grad()
def time_fwd(model, q=1, ctx=150, steps=60, reps=3):
    ts = []
    for _ in range(reps):
        x = torch.randint(1000, 20000, (1, ctx), device=dev)
        o = model(x, use_cache=True); pkv = o.past_key_values
        for s in range(steps + 10):
            inp = torch.randint(1000, 20000, (1, q), device=dev)
            sync_all(); t = time.perf_counter()
            o = model(inp, past_key_values=pkv, use_cache=True)
            sync_all(); dt = time.perf_counter() - t
            pkv = o.past_key_values
            if s >= 10: ts.append(dt)
    return float(np.median(ts))

t_d = time_fwd(draft, 1)
t_t = time_fwd(target, 1)
print(f"draft  (1B) decode step: {t_d*1000:.2f} ms")
print(f"target (8B) decode step: {t_t*1000:.2f} ms")
print(f"c = t_draft / t_target = {t_d/t_t:.3f}")

# Verification cost for gamma+1 tokens (should be close to one decode step if memory-bound).
# gamma=1 is included too, since dynamic policies can draft a single token.
ver = {g: time_fwd(target, g+1) for g in range(1, 9)}
print({g: f"{v*1000:.1f} ms" for g, v in ver.items()})

json.dump(dict(t_draft=t_d, t_target=t_t, c=t_d/t_t, t_target_verify={str(g): v for g, v in ver.items()},
               setup=dict(target=TARGET, draft=DRAFT, dtype=str(DTYPE), max_memory=MAX_MEMORY,
                          gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])),
          open(OUT/"timing.json", "w"), indent=1)

In [ ]:
# Free the GPUs; Part B is pure CPU.
del target, draft
gc.collect(); torch.cuda.empty_cache()
gpu_report("after cleanup:")
print(sorted(p.name for p in OUT.iterdir()))

# Part B: Day 3 (trace simulator, fixed-k baseline, entropy policy, oracle)

Simulates speculative decoding *offline* from the Part A traces (no GPU needed). If you start a fresh session for this part, add the saved `day2_8b` folder as a Kaggle dataset and point `DATA_DIR` at `/kaggle/input/<your-dataset>/`.

**Why a trace simulator is exact for greedy decoding:** within a draft window, the draft's tokens are conditioned on its own previous drafts. Those coincide with the target's tokens up to the first mismatch, and everything after the first mismatch is rejected anyway. So the number accepted = number of leading matches in the window.

**Cost model:** one round with `g` drafted tokens costs `g*c + 1` (target forward = 1, draft forward = `c`) and yields `accepted + 1` tokens (the +1 is the correction or bonus token from the target). Plain decoding costs 1 per token, so `speed-up = tokens emitted / total cost`.

In [ ]:
import json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR    = Path("/kaggle/working/day2_8b")      # folder with draft_traces.json + timing.json from Part A
RES_DIR     = DATA_DIR / "day3"; RES_DIR.mkdir(parents=True, exist_ok=True)
TRACE_PATH  = DATA_DIR / "draft_traces.json"
TIMING_PATH = DATA_DIR / "timing.json"
K_RANGE     = range(2, 9)                          # fixed windows k = 2..8
GAMMA_MAX   = 8                                    # draft cap for dynamic policies (same as max k, for a fair comparison)
SEED        = 0
random.seed(SEED); np.random.seed(SEED)

def load_traces(path):
    traces = []
    for r in json.load(open(path)):
        tgt = np.asarray(r["target_ids"]); drf = np.asarray(r["draft_argmax"])
        H = np.asarray(r["entropy"], dtype=float)
        assert len(tgt) == len(drf) == len(H), f"length mismatch in sentence {r.get('id')}"
        match = (tgt == drf)
        assert np.array_equal(match, np.asarray(r["match"], bool)), f"match mismatch in sentence {r.get('id')}"
        traces.append(dict(id=r.get("id"), n=len(tgt), match=match, H=H, target_ids=tgt,
                           p_target=np.asarray(r.get("p_target", []), float), tokens=None))
    return traces

assert TRACE_PATH.exists() and TIMING_PATH.exists(), f"missing Part A outputs in {DATA_DIR}"
traces = load_traces(TRACE_PATH)

# Cost model. c = draft step / target step. Optionally use the measured verification cost for gamma+1 tokens too.
T = json.load(open(TIMING_PATH)); C = T["c"]
gs = sorted(int(g) for g in T["t_target_verify"])
vs = [max(1.0, T["t_target_verify"][str(g)] / T["t_target"]) for g in gs]   # clip: verify can't beat one decode step (dips are timing noise)
VERIFY = lambda g: 1.0 if g == 0 else float(np.interp(g, gs, vs))
print(f"c = {C:.3f} (measured) | verify cost / decode step for gamma={gs[0]}..{gs[-1]}: {[round(v, 3) for v in vs]}")

n_tok = sum(s["n"] for s in traces)
print(f"{len(traces)} sentences, {n_tok} target tokens, per-token draft/target match rate = {np.concatenate([s['match'] for s in traces]).mean():.3f}")

## Simulator

A **policy** is a callable `policy(sent, p, g) -> bool` that answers: *"we're at position `p` and have already drafted `g` tokens in this window; stop drafting now?"* It is called before each additional draft token.

In [ ]:
def simulate(traces, policy, gamma_max, c, verify=None):
    rounds = drafted = accepted = rejections = emitted = 0
    cost = 0.0
    draft_lens = []
    for s in traces:
        n, match = s["n"], s["match"]
        p = 0
        while p < n:
            r = n - p                              # tokens still to produce
            cap = min(gamma_max, r)
            g = 0
            while g < cap and not policy(s, p, g):  # draft one more token
                g += 1
            a = 0                                  # leading matches = accepted
            while a < g and match[p + a]:
                a += 1
            gained = min(a + 1, r)                 # + correction/bonus token from target
            rounds += 1; drafted += g; accepted += a
            rejections += int(a < g); emitted += gained
            cost += g * c + (verify(g) if verify else 1.0)
            draft_lens.append(g)
            p += gained
    return dict(rounds=rounds, drafted=drafted, accepted=accepted, rejections=rejections,
                emitted=emitted, cost=cost, draft_lens=draft_lens)

def summarize(res):
    return dict(
        acceptance_rate=res["accepted"] / max(res["drafted"], 1),                         # accepted / drafted
        alpha_hat=res["accepted"] / max(res["accepted"] + res["rejections"], 1),          # per-token accept prob (geometric MLE)
        tokens_per_verif=res["emitted"] / res["rounds"],
        mean_draft_len=res["drafted"] / res["rounds"],
        speedup=res["emitted"] / res["cost"],
    )

def formula_speedup(alpha, gamma, c):
    # f = (1 - a^(g+1)) / ((1 - a)(g c + 1)),  i.i.d. acceptance assumption
    exp_tokens = (gamma + 1) if alpha >= 1 else (1 - alpha ** (gamma + 1)) / (1 - alpha)
    return exp_tokens / (gamma * c + 1)

# ---- policies ----
class FixedK:
    def __init__(self, k): self.k = k
    def __call__(self, s, p, g): return g >= self.k

class EntropyPolicy:
    # AdaEDL/SVIP-style: after drafting a token, stop if 1 - sqrt(beta * H) < lam.
    # NOTE: beta is the entropy scale hyperparameter (called gamma in SVIP), NOT the draft length.
    def __init__(self, lam, beta=1.0): self.lam, self.beta = lam, beta
    def __call__(self, s, p, g):
        if g == 0: return False                    # always draft at least one token
        H = max(s["H"][p + g - 1], 0.0)            # entropy of the last drafted token
        return 1.0 - math.sqrt(self.beta * H) < self.lam

class Oracle:
    # Upper bound: knows the future; drafts exactly the leading run of matches, never wastes a draft token.
    def __call__(self, s, p, g): return not s["match"][p + g]

class NoDraft:
    def __call__(self, s, p, g): return True

In [ ]:
# ---- sanity checks on tiny hand-made traces ----
def _t(match):
    m = np.array(match, bool); return dict(id=0, n=len(m), match=m, H=np.zeros(len(m)), tokens=None)

# all match, k=4, c=0: 10 tokens -> rounds emit 5,5 => 2 rounds
r = simulate([_t([1]*10)], FixedK(4), 4, 0.0); assert r["rounds"] == 2 and r["emitted"] == 10, r
# nothing matches: every round drafts k, accepts 0, emits 1
r = simulate([_t([0]*6)], FixedK(3), 3, 0.0); assert r["rounds"] == 6 and r["accepted"] == 0 and r["emitted"] == 6, r
# no drafting == speed-up 1
r = simulate([_t([1]*7)], NoDraft(), 8, 0.3); assert abs(summarize(r)["speedup"] - 1) < 1e-9
# oracle never rejects
r = simulate(traces, Oracle(), GAMMA_MAX, C); assert r["rejections"] == 0 and r["emitted"] == n_tok
# every policy emits exactly the number of target tokens
for pol in [FixedK(5), EntropyPolicy(0.5)]:
    assert simulate(traces, pol, GAMMA_MAX, C)["emitted"] == n_tok
print("sanity checks passed")

## 1. Fixed-k baseline (k = 2..8)
Simulated numbers vs. the closed-form formula `f = (1 - a^(g+1)) / ((1 - a)(g c + 1))`. The formula assumes i.i.d. acceptance, so the gap between the two columns tells you how non-i.i.d. your traces are.

In [ ]:
rows = []
for k in K_RANGE:
    res = simulate(traces, FixedK(k), k, C); m = summarize(res)
    m2 = summarize(simulate(traces, FixedK(k), k, C, VERIFY))
    rows.append(dict(k=k, **m, formula_speedup=formula_speedup(m["alpha_hat"], k, C), speedup_measured_verify=m2["speedup"]))
fixed_df = pd.DataFrame(rows).set_index("k")
print(f"c = {C:.3f}")
fixed_df.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(fixed_df.index, fixed_df.acceptance_rate, "o-", label="accepted / drafted")
ax[0].plot(fixed_df.index, fixed_df.alpha_hat, "s--", label="alpha_hat (per-token)")
ax[0].set(xlabel="k", ylabel="rate", title="Acceptance vs k"); ax[0].legend()
ax[1].plot(fixed_df.index, fixed_df.tokens_per_verif, "o-")
ax[1].set(xlabel="k", ylabel="tokens / verification", title="Accepted tokens per verification (incl. +1)")
ax[2].plot(fixed_df.index, fixed_df.speedup, "o-", label="simulated")
ax[2].plot(fixed_df.index, fixed_df.formula_speedup, "s--", label="formula (iid)")
ax[2].plot(fixed_df.index, fixed_df.speedup_measured_verify, "^:", label="simulated, measured verify cost")
ax[2].axhline(1, color="gray", lw=0.8); ax[2].set(xlabel="k", ylabel="speed-up", title="Speed-up vs k"); ax[2].legend()
plt.tight_layout(); plt.show()
best_k = int(fixed_df.speedup.idxmax()); print("best fixed k:", best_k, f"-> {fixed_df.speedup.max():.3f}x")

## 2. Entropy-only policy (AdaEDL/SVIP rule)
Stop when `1 - sqrt(beta * H) < lambda`. To avoid tuning on the same data we report, sentences are split into dev/test, `lambda` and `beta` are swept on dev, and the chosen setting is reported on test.

In [ ]:
idx = np.random.default_rng(SEED).permutation(len(traces))
dev_set  = [traces[i] for i in idx[: len(idx)//2]]
test_set = [traces[i] for i in idx[len(idx)//2 :]]

LAMS  = np.round(np.linspace(-1.5, 0.95, 40), 3)
BETAS = [0.5, 1.0, 2.0]
sweep = []
for beta in BETAS:
    for lam in LAMS:
        m = summarize(simulate(dev_set, EntropyPolicy(lam, beta), GAMMA_MAX, C))
        sweep.append(dict(beta=beta, lam=lam, **m))
sweep_df = pd.DataFrame(sweep)
best = sweep_df.loc[sweep_df.speedup.idxmax()]
print("best on dev:\n", best.round(3).to_string())

plt.figure(figsize=(6, 4))
for beta, g in sweep_df.groupby("beta"): plt.plot(g.lam, g.speedup, "o-", ms=3, label=f"beta={beta}")
plt.axhline(summarize(simulate(dev_set, FixedK(best_k), best_k, C))["speedup"], color="gray", ls="--", label=f"fixed k={best_k}")
plt.xlabel("lambda"); plt.ylabel("speed-up (dev)"); plt.title("Entropy policy: lambda sweep"); plt.legend(); plt.show()

## 3. Oracle (upper bound)
Drafts exactly the run of tokens the target will accept (capped at `GAMMA_MAX`), so no draft compute is wasted and no rejections occur. Also shown uncapped (cap 32) to see how much the cap itself costs.

In [ ]:
oracle_res = simulate(test_set, Oracle(), GAMMA_MAX, C)
oracle_unc = simulate(test_set, Oracle(), 32, C)
print("oracle (cap 8): ", {k: round(v, 3) for k, v in summarize(oracle_res).items()})
print("oracle (cap 32):", {k: round(v, 3) for k, v in summarize(oracle_unc).items()})

## 4. Summary table (test split)
Fixed-k and entropy hyperparameters are chosen on dev; all rows below are on the held-out test sentences.

In [ ]:
best_k_dev = max(K_RANGE, key=lambda k: summarize(simulate(dev_set, FixedK(k), k, C))["speedup"])
policies = {
    "no speculation":            (NoDraft(), GAMMA_MAX),
    f"fixed k={best_k_dev} (dev-best)": (FixedK(best_k_dev), best_k_dev),
    f"entropy (lam={best.lam:.2f}, beta={best.beta})": (EntropyPolicy(best.lam, best.beta), GAMMA_MAX),
    "oracle (cap 8)":            (Oracle(), GAMMA_MAX),
}
rows = {}
for name, (pol, cap) in policies.items():
    res = simulate(test_set, pol, cap, C); rows[name] = summarize(res)
    rows[name]["speedup_measured_verify"] = summarize(simulate(test_set, pol, cap, C, VERIFY))["speedup"]
summary_df = pd.DataFrame(rows).T
display(summary_df.round(3))

summary_df.to_csv(RES_DIR / "day3_summary_test_8b.csv")
fixed_df.to_csv(RES_DIR / "day3_fixed_k_full_8b.csv")
sweep_df.to_csv(RES_DIR / "day3_entropy_sweep_dev_8b.csv")
ax = summary_df.speedup.plot.barh(figsize=(7, 3)); ax.axvline(1, color="gray", lw=0.8)
ax.set_xlabel("speed-up (test)"); ax.set_title("Speed-up per policy (8B target / 1B draft)"); plt.tight_layout(); plt.show()

## Notes for the write-up
- Quote **simulated** speed-ups as primary and the formula as a cross-check; Day 4's real draft-and-verify loop validates them.
- `c` was measured with the 8B target layer-split over 2x T4 (sequential, with a GPU0 -> GPU1 hop per step) and the 1B draft on one GPU. On a single large GPU (A100/H100) the target step would be cheaper per layer-hop, so `c` would shift; say this explicitly.
- `c` must be the per-step decoding cost ratio (with KV cache), not prefill time. A wrong `c` changes every speed-up number.
- The gap `oracle - entropy` is the headroom for boundary-aware policies.
- Day 4 hook: add per-position arrays to each trace dict (e.g. `s["is_word_end"]`) and write policies with the same `(s, p, g) -> bool` signature.
- Compare against your 3B run: only the target changed, so differences in agreement rate, entropy, and `c` are attributable to the 8B target.

In [ ]:
# Bundle everything (outputs + day3 CSVs) into one zip in /kaggle/working so it shows up under the notebook's Output tab.
import shutil
zip_path = shutil.make_archive("/kaggle/working/day2_day3_8b_artifacts", "zip", DATA_DIR)
print("saved", zip_path)